# Inventory Quote Model

This notebook connects three pieces of the market-making stack:

1. standalone passive quote economics,
2. fair-value edge and adverse-selection residuals,
3. inventory-aware quote distance and quote/no-quote decisions.

The goal is not to force every quote bucket positive. The goal is to identify what actually moves expected value and which assumptions require event-level validation before deployment.

## Research Definition

For side `s` and quote distance `d`, define a passive quote at the current reference price `S_t`:

$$
Q_t(s,d)=
\begin{cases}
S_t\exp(-d/10^4), & s=bid \\
S_t\exp(+d/10^4), & s=ask
\end{cases}
$$

The standalone quote score is the expected markout after a fill, net of maker fee `c`:

$$
EV^{standalone}_{t,s,d}
=\Pr(T_{t,s,d}=1\mid x_t)\left(\mathbb{E}[m_{t,s,d}\mid T=1,x_t]-c\right)
$$

The fee lever is therefore limited. If the mean touched markout is negative, the required maker rebate per fill is:

$$
rebate^*_{s,d}=\max(0,-\bar m_{s,d}^{touch})
$$

For inventory-aware quoting, the fair-value model predicts a 30-minute edge `f_t` in bps. The fair-value edge if filled is:

$$
e^{FV}_{bid,d}=d+f_t,\qquad e^{FV}_{ask,d}=d-f_t
$$

We model toxicity as the residual after fair value:

$$
\epsilon^{tox}_{t,s,d}=m_{t,s,d}-e^{FV}_{t,s,d}
$$

With inventory `q`, side sign `\eta_s=+1` for bid and `-1` for ask, and risk charge `\lambda_t`, the inventory quote score is:

$$
EV_{t,s,d,q}=p^{touch}_{t,s,d}\left(e^{FV}_{t,s,d}+\hat\epsilon^{tox}_{t,s,d}-c-\eta_s q\lambda_t-h\right)
$$

where `h` is a model-risk haircut. The haircut is not a substitute for better data; it prevents the inventory quote table from treating noisy edge estimates as executable edge.


Positive standalone EV means the raw fill model can clear zero without fair value or inventory help. Positive inventory-aware EV means the quote is acceptable after fair value, toxicity, fees, and inventory pressure.

In [ ]:
from __future__ import annotations

import os
import sys
from datetime import datetime
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

if "notebooks" not in sys.path:
    sys.path.append("notebooks")

from advanced_features import (
    FEATURE_DATASET_KEYS,
    available_dataset_dates,
    build_feature_set,
    build_targets,
    discover_datasets,
)

ROOT = Path(os.environ.get("MODL_WS_NORMALIZED_DIR", "/mnt/burner-archive/ws_normalized")).expanduser()
FEATURE_ROOT = Path(os.environ.get("MODL_WS_FEATURE_DIR", "/mnt/burner-archive/ws_features")).expanduser()
BAR_MINUTES = int(os.environ.get("MODL_BAR_MINUTES", "5"))
HORIZONS = tuple(int(value) for value in os.environ.get("MODL_HORIZONS", "5,15,30").split(","))
DATE_FILTER = [value.strip() for value in os.environ.get("MODL_INVENTORY_DATES", "").split(",") if value.strip()]

INCLUDE_PARTIAL_DATES = True
FAIR_VALUE_TARGET = os.environ.get("MODL_FAIR_VALUE_TARGET", "target_fir_return_entry5m_wait0m_exit30m")
QUOTE_DISTANCES_BPS = tuple(float(value) for value in os.environ.get("MODL_QUOTE_DISTANCES_BPS", "2,5,10,20,30,50,75,100").split(","))
TOUCH_HORIZON_MINUTES = int(os.environ.get("MODL_TOUCH_HORIZON_MINUTES", "15"))
MARKOUT_MINUTES = int(os.environ.get("MODL_MARKOUT_MINUTES", "30"))
SENSITIVITY_TOUCH_HORIZONS = tuple(int(value) for value in os.environ.get("MODL_TOUCH_SENSITIVITY_MINUTES", "5,10,15,30").split(","))
SENSITIVITY_MARKOUT_HORIZONS = tuple(int(value) for value in os.environ.get("MODL_MARKOUT_SENSITIVITY_MINUTES", "5,15,30,60").split(","))
STANDALONE_THRESHOLDS_BPS = tuple(float(value) for value in os.environ.get("MODL_STANDALONE_THRESHOLDS_BPS", "-5,-2,0,1,2,5").split(","))

MIN_FEATURE_OBS = int(os.environ.get("MODL_INVENTORY_MIN_FEATURE_OBS", "50"))
MIN_TRAIN_OBS = int(os.environ.get("MODL_INVENTORY_MIN_TRAIN_OBS", "250"))
MAKER_FEE_BPS = float(os.environ.get("MODL_MAKER_FEE_BPS", "0.0"))
LOGIT_C = float(os.environ.get("MODL_FILL_LOGIT_C", "0.25"))
RIDGE_ALPHA = float(os.environ.get("MODL_RIDGE_ALPHA", "25.0"))
INVENTORY_UNITS = tuple(float(value) for value in os.environ.get("MODL_INVENTORY_UNITS", "-2,-1,0,1,2").split(","))
INVENTORY_RISK_BPS_PER_UNIT = float(os.environ.get("MODL_INVENTORY_RISK_BPS_PER_UNIT", "5.0"))
MIN_QUOTE_SCORE_BPS = float(os.environ.get("MODL_MIN_QUOTE_SCORE_BPS", "0.0"))
MODEL_RISK_HAIRCUT_BPS = float(os.environ.get("MODL_MODEL_RISK_HAIRCUT_BPS", "10.0"))
VOL_LOOKBACK_BARS = int(os.environ.get("MODL_VOL_LOOKBACK_BARS", "6"))
SAVE_OUTPUTS = False

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 280)
pd.set_option("display.max_colwidth", 240)

ROOT, BAR_MINUTES, QUOTE_DISTANCES_BPS, TOUCH_HORIZON_MINUTES, MARKOUT_MINUTES

## Date Inventory And Panel Build

The notebook includes full and partial dates so newly synced data is automatically included. Partial dates are flagged in the inventory table.

In [ ]:
date_key_map = available_dataset_dates(ROOT)
date_inventory = pd.DataFrame(
    [
        {
            "date": date,
            "dataset_count": len(keys),
            "missing_required": sorted(set(FEATURE_DATASET_KEYS) - set(keys)),
            "is_full_dataset": set(FEATURE_DATASET_KEYS).issubset(keys),
        }
        for date, keys in sorted(date_key_map.items())
    ]
)
if DATE_FILTER:
    date_inventory = date_inventory[date_inventory["date"].isin(DATE_FILTER)].copy()
if not INCLUDE_PARTIAL_DATES:
    date_inventory = date_inventory[date_inventory["is_full_dataset"]].copy()

feature_by_date: dict[str, pd.DataFrame] = {}
target_by_date: dict[str, pd.DataFrame] = {}
model_by_date: dict[str, pd.DataFrame] = {}
build_rows: list[dict[str, object]] = []
build_errors: list[dict[str, object]] = []

for date in date_inventory["date"].tolist():
    date_tag = datetime.strptime(date, "%Y-%m-%d").strftime("%y-%m-%d")
    datasets = discover_datasets(ROOT, date_tag)
    try:
        feature_set = build_feature_set(datasets, horizons=HORIZONS, bar_minutes=BAR_MINUTES)
        features = feature_set.feature_matrix.copy()
        targets = build_targets(feature_set.reference_price, feature_set.term_structure, HORIZONS, bar_minutes=BAR_MINUTES)
        model = pd.concat([features, targets], axis=1).sort_index()

        feature_by_date[date] = features
        target_by_date[date] = targets
        model_by_date[date] = model
        build_rows.append(
            {
                "date": date,
                "is_full_dataset": set(FEATURE_DATASET_KEYS).issubset(date_key_map[date]),
                "datasets": len(datasets),
                "rows": len(model),
                "feature_columns": features.shape[1],
                "target_columns": targets.shape[1],
                "reference_obs": int(model["reference_price"].notna().sum()),
            }
        )
    except Exception as exc:  # noqa: BLE001 - research notebook should report and continue
        build_errors.append({"date": date, "error_type": type(exc).__name__, "error": str(exc)})

if not model_by_date:
    raise RuntimeError("No inventory quote dates built successfully")

build_summary = pd.DataFrame(build_rows).sort_values("date")
build_errors = pd.DataFrame(build_errors)
model_panel = pd.concat(model_by_date, names=["date", "minute"]).sort_index()

display(date_inventory)
display(build_summary)
if not build_errors.empty:
    display(build_errors)
model_panel.shape

## Passive Quote Labels And Baseline EV

The first table is the raw passive quote baseline. It assumes a quote is filled if the future bar-level path touches the quote within the touch horizon. That is deliberately conservative because it does not yet model queue position or fast cancels.

In [ ]:
def future_min(series: pd.Series, periods: int) -> pd.Series:
    return series.shift(-1).iloc[::-1].rolling(periods, min_periods=periods).min().iloc[::-1]


def future_max(series: pd.Series, periods: int) -> pd.Series:
    return series.shift(-1).iloc[::-1].rolling(periods, min_periods=periods).max().iloc[::-1]


def make_quote_labels(
    models: dict[str, pd.DataFrame],
    distances_bps: tuple[float, ...],
    touch_minutes: int,
    markout_minutes: int,
    maker_fee_bps: float = MAKER_FEE_BPS,
) -> pd.DataFrame:
    touch_periods = max(1, int(round(touch_minutes / BAR_MINUTES)))
    markout_periods = max(1, int(round(markout_minutes / BAR_MINUTES)))
    frames: list[pd.DataFrame] = []
    for date, model in models.items():
        reference = model["reference_price"].astype(float)
        fmin = future_min(reference, touch_periods)
        fmax = future_max(reference, touch_periods)
        mark_price = reference.shift(-markout_periods)
        for side, side_sign in [("bid", 1.0), ("ask", -1.0)]:
            for distance_bps in distances_bps:
                quote_price = reference * np.exp(-side_sign * distance_bps / 10_000.0)
                if side == "bid":
                    touch = fmin <= quote_price
                    markout_bps = 10_000.0 * np.log(mark_price / quote_price)
                else:
                    touch = fmax >= quote_price
                    markout_bps = 10_000.0 * np.log(quote_price / mark_price)
                frame = pd.DataFrame(index=pd.MultiIndex.from_product([[date], model.index], names=["date", "minute"]))
                frame["side"] = side
                frame["side_sign"] = side_sign
                frame["quote_distance_bps"] = float(distance_bps)
                frame["quote_price"] = quote_price.to_numpy()
                frame["touch"] = touch.astype(float).to_numpy()
                frame["markout_bps"] = markout_bps.to_numpy()
                frame["realized_quote_value_bps"] = np.where(frame["touch"] == 1.0, frame["markout_bps"] - maker_fee_bps, 0.0)
                frames.append(frame)
    return pd.concat(frames).sort_index().replace([np.inf, -np.inf], np.nan)


def quote_bucket_summary(labels: pd.DataFrame) -> pd.DataFrame:
    base = labels.dropna(subset=["touch"]).copy()
    summary = (
        base.groupby(["side", "quote_distance_bps"], as_index=False)
        .agg(
            rows=("touch", "count"),
            touch_rate=("touch", "mean"),
            realized_quote_value_bps=("realized_quote_value_bps", "mean"),
        )
    )
    touched = (
        base[base["touch"] == 1.0]
        .groupby(["side", "quote_distance_bps"], as_index=False)
        .agg(
            touched_rows=("touch", "count"),
            mean_markout_if_touched=("markout_bps", "mean"),
            adverse_touch_share=("markout_bps", lambda value: (value < 0).mean()),
        )
    )
    summary = summary.merge(touched, on=["side", "quote_distance_bps"], how="left")
    summary["required_rebate_per_fill_bps"] = (-summary["mean_markout_if_touched"]).clip(lower=0.0)
    summary["posted_ev_gap_to_zero_bps"] = (-summary["realized_quote_value_bps"]).clip(lower=0.0)
    return summary.sort_values(["side", "quote_distance_bps"]).reset_index(drop=True)


quote_labels = make_quote_labels(model_by_date, QUOTE_DISTANCES_BPS, TOUCH_HORIZON_MINUTES, MARKOUT_MINUTES)
quote_baseline = quote_bucket_summary(quote_labels)
display(quote_baseline)
quote_labels.shape

## Standalone EV Sensitivity

This scan answers what can make raw quote buckets positive before fair value and inventory. Wider quotes, shorter toxic fill windows, and different markout horizons can improve average EV, but low fill counts must be treated as fragile until event-level queue simulation confirms them.

In [ ]:
sensitivity_rows: list[dict[str, object]] = []
positive_bucket_frames: list[pd.DataFrame] = []

for touch_minutes in SENSITIVITY_TOUCH_HORIZONS:
    for markout_minutes in SENSITIVITY_MARKOUT_HORIZONS:
        labels = make_quote_labels(model_by_date, QUOTE_DISTANCES_BPS, touch_minutes, markout_minutes)
        summary = quote_bucket_summary(labels)
        positive = summary[summary["realized_quote_value_bps"] > 0].copy()
        if not positive.empty:
            positive["touch_horizon_minutes"] = touch_minutes
            positive["markout_minutes"] = markout_minutes
            positive_bucket_frames.append(positive)
        best = summary.loc[summary["realized_quote_value_bps"].idxmax()]
        sensitivity_rows.append(
            {
                "touch_horizon_minutes": touch_minutes,
                "markout_minutes": markout_minutes,
                "positive_bucket_count": int((summary["realized_quote_value_bps"] > 0).sum()),
                "best_side": best["side"],
                "best_distance_bps": best["quote_distance_bps"],
                "best_touch_rate": best["touch_rate"],
                "best_realized_quote_value_bps": best["realized_quote_value_bps"],
                "best_mean_markout_if_touched": best["mean_markout_if_touched"],
            }
        )

sensitivity_summary = pd.DataFrame(sensitivity_rows).sort_values(["touch_horizon_minutes", "markout_minutes"])
positive_buckets = pd.concat(positive_bucket_frames, ignore_index=True) if positive_bucket_frames else pd.DataFrame()

display(sensitivity_summary)
if not positive_buckets.empty:
    display(positive_buckets.sort_values("realized_quote_value_bps", ascending=False).head(30))

## Feature Policy

The feature policy matches the fair-value and fill notebooks. Raw price levels are excluded so the model learns state, flow, volatility, basis, funding, and microstructure relationships rather than the BTC price level.

In [ ]:
ROLLING_SUFFIXES = (
    f"_diff_{BAR_MINUTES}m",
    "_mean_5m",
    "_z_5m",
    "_mean_15m",
    "_z_15m",
    "_mean_30m",
    "_z_30m",
)


def base_feature_name(feature: str) -> str:
    for suffix in ROLLING_SUFFIXES:
        if feature.endswith(suffix):
            return feature[: -len(suffix)]
    return feature


def feature_family(feature: str) -> str:
    base = base_feature_name(feature)
    rules = [
        ("hawkes_bsi", ("hawkes_bsi_",)),
        ("trade_flow", ("trade_flow_imbalance_",)),
        ("trade_activity", ("trade_volume_", "trade_trade_count_", "trade_vwap_")),
        ("book_microstructure", ("book_",)),
        ("cross_venue", ("cross_",)),
        ("spread_mid_arbitrage", ("mid_", "spread_")),
        ("realized_vol", ("rv_", "bpv_", "jump_")),
        ("returns", ("ret_",)),
        ("term_structure", ("iv_", "term_", "short_iv_decimal")),
        ("option_smile", ("smile_",)),
        ("futures_basis", ("basis_",)),
        ("funding", ("estimated_funding_rate",)),
    ]
    for family, prefixes in rules:
        if base.startswith(prefixes):
            return family
    return "other"


def is_disallowed_level(feature: str) -> bool:
    base = base_feature_name(feature)
    if base in {"reference_price", "log_price", "median_index_price", "mid_hibachi_minus_hyperliquid"}:
        return True
    if base.startswith(("book_mid_", "trade_vwap_")):
        return True
    if base.endswith(("_count", "_updates")) and not base.startswith("trade_trade_count_"):
        return True
    if base in {"active_options", "option_tick_count"}:
        return True
    return False


included_families = {
    "hawkes_bsi",
    "trade_flow",
    "trade_activity",
    "book_microstructure",
    "cross_venue",
    "spread_mid_arbitrage",
    "realized_vol",
    "returns",
    "term_structure",
    "option_smile",
    "futures_basis",
    "funding",
}

numeric_features = [column for column in model_panel.select_dtypes(include=[np.number]).columns if not column.startswith("target_")]
state_features = [
    column
    for column in numeric_features
    if feature_family(column) in included_families
    and not is_disallowed_level(column)
    and model_panel[column].replace([np.inf, -np.inf], np.nan).notna().sum() >= MIN_FEATURE_OBS
    and model_panel[column].nunique(dropna=True) > 1
]
quote_features = ["side_sign", "quote_distance_bps"]
model_features = quote_features + state_features

feature_catalog = pd.DataFrame(
    {
        "feature": state_features,
        "base_feature": [base_feature_name(column) for column in state_features],
        "family": [feature_family(column) for column in state_features],
        "is_rolling_transform": [base_feature_name(column) != column for column in state_features],
    }
)
display(feature_catalog.groupby("family").size().rename("feature_count").sort_values(ascending=False).to_frame())
len(model_features)

## Model Table

The quote table repeats each market state across side and quote-distance candidates. We also add a simple realized volatility scale used by the inventory risk charge.

In [ ]:
def realized_vol_scale_by_date(models: dict[str, pd.DataFrame]) -> pd.Series:
    frames: list[pd.Series] = []
    for date, model in models.items():
        reference = model["reference_price"].astype(float)
        ret_bps = 10_000.0 * np.log(reference / reference.shift(1))
        vol_bps = ret_bps.rolling(VOL_LOOKBACK_BARS, min_periods=2).std()
        median_vol = vol_bps.replace([np.inf, -np.inf], np.nan).median()
        if not np.isfinite(median_vol) or median_vol <= 0:
            median_vol = 1.0
        scale = (vol_bps / median_vol).clip(lower=0.25, upper=4.0).fillna(1.0)
        frames.append(pd.Series(scale.to_numpy(), index=pd.MultiIndex.from_product([[date], model.index], names=["date", "minute"]), name="vol_scale"))
    return pd.concat(frames).sort_index()


feature_rows = model_panel.reindex(columns=state_features)
vol_scale = realized_vol_scale_by_date(model_by_date)
quote_model_table = quote_labels.join(feature_rows, how="left")
quote_model_table["vol_scale"] = vol_scale.reindex(quote_model_table.index).to_numpy()
quote_model_table = quote_model_table.dropna(subset=["touch", "markout_bps", "side_sign", "quote_distance_bps"])

coverage = pd.DataFrame(
    {
        "non_null": quote_model_table[model_features + ["touch", "markout_bps", "vol_scale"]].notna().sum(),
        "coverage": quote_model_table[model_features + ["touch", "markout_bps", "vol_scale"]].notna().mean(),
    }
).sort_values("coverage")
display(coverage.head(30))
quote_model_table.shape

## Models

We fit three models in walk-forward form:

1. fair-value ridge model for `f_t`,
2. touch logistic model for fill probability,
3. raw markout and toxicity-residual ridge models for quote value.

The raw markout model is the standalone strategy candidate. The residual model is used when fair value and inventory are explicitly part of the quote score.

In [ ]:
def make_fair_value_model(alpha: float = RIDGE_ALPHA) -> Pipeline:
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=alpha)),
        ]
    )


def make_touch_model() -> Pipeline:
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("logit", LogisticRegression(C=LOGIT_C, class_weight="balanced", max_iter=2_000)),
        ]
    )


def make_ridge_model(alpha: float = RIDGE_ALPHA) -> Pipeline:
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=alpha)),
        ]
    )


def available_feature_columns(frame: pd.DataFrame, features: list[str], min_obs: int = MIN_FEATURE_OBS) -> list[str]:
    return [
        column
        for column in features
        if column in frame.columns
        and frame[column].replace([np.inf, -np.inf], np.nan).notna().sum() >= min_obs
        and frame[column].nunique(dropna=True) > 1
    ]


def valid_fair_value_frame(frame: pd.DataFrame, features: list[str]) -> tuple[pd.DataFrame, pd.Series]:
    if FAIR_VALUE_TARGET not in frame.columns:
        raise KeyError(f"FAIR_VALUE_TARGET is missing: {FAIR_VALUE_TARGET}")
    table = frame.reindex(columns=features + [FAIR_VALUE_TARGET]).replace([np.inf, -np.inf], np.nan).dropna(subset=[FAIR_VALUE_TARGET])
    usable = available_feature_columns(table, features, min_obs=MIN_FEATURE_OBS)
    return table[usable], table[FAIR_VALUE_TARGET].mul(10_000.0)


def side_fair_value_edge_bps(frame: pd.DataFrame, edge_col: str = "fair_value_edge_bps") -> pd.Series:
    edge = frame[edge_col].astype(float)
    distance = frame["quote_distance_bps"].astype(float)
    value = np.where(frame["side"].eq("bid"), distance + edge, distance - edge)
    return pd.Series(value, index=frame.index, name="fair_value_edge_if_filled_bps")


def touch_metrics(actual: pd.Series, predicted: pd.Series) -> dict[str, float]:
    pair = pd.concat([actual.rename("actual"), predicted.rename("predicted")], axis=1).dropna()
    if pair.empty:
        return {"n": 0, "touch_rate": np.nan, "brier": np.nan, "log_loss": np.nan, "auc": np.nan}
    has_two_classes = pair["actual"].nunique() == 2
    return {
        "n": int(len(pair)),
        "touch_rate": float(pair["actual"].mean()),
        "brier": float(brier_score_loss(pair["actual"], pair["predicted"])),
        "log_loss": float(log_loss(pair["actual"], pair["predicted"], labels=[0.0, 1.0])),
        "auc": float(roc_auc_score(pair["actual"], pair["predicted"])) if has_two_classes else np.nan,
    }


def regression_metrics(actual: pd.Series, predicted: pd.Series, prefix: str) -> dict[str, float]:
    pair = pd.concat([actual.rename("actual"), predicted.rename("predicted")], axis=1).replace([np.inf, -np.inf], np.nan).dropna()
    if pair.empty:
        return {f"{prefix}_n": 0, f"{prefix}_rmse_bps": np.nan, f"{prefix}_mae_bps": np.nan, f"{prefix}_spearman": np.nan}
    err = pair["predicted"] - pair["actual"]
    return {
        f"{prefix}_n": int(len(pair)),
        f"{prefix}_rmse_bps": float(np.sqrt(np.mean(np.square(err)))),
        f"{prefix}_mae_bps": float(np.mean(np.abs(err))),
        f"{prefix}_spearman": float(pair["actual"].corr(pair["predicted"], method="spearman")) if pair["actual"].nunique() > 1 and pair["predicted"].nunique() > 1 else np.nan,
    }

## Walk-Forward Quote Models

Each test date is scored using only earlier dates. With the current data inventory this mainly gives one usable out-of-sample day, so the results are directional research evidence, not production proof.

In [ ]:
walk_forward_frames: list[pd.DataFrame] = []
walk_forward_rows: list[dict[str, object]] = []
ordered_dates = sorted(model_by_date)

for test_date in ordered_dates:
    train_dates = [date for date in ordered_dates if date < test_date]
    test_frame = quote_model_table.xs(test_date, level="date", drop_level=False).copy()
    if not train_dates:
        walk_forward_rows.append({"test_date": test_date, "status": "skipped_no_prior_dates", "train_obs": 0, "test_obs": len(test_frame)})
        continue

    train_quote_frame = quote_model_table.loc[pd.IndexSlice[train_dates, :], :].copy()
    train_model_frame = pd.concat([model_by_date[date] for date in train_dates]).sort_index()
    train_model_frame.index = pd.MultiIndex.from_tuples(
        [(date, minute) for date in train_dates for minute in model_by_date[date].index],
        names=["date", "minute"],
    )
    test_model_frame = model_by_date[test_date].copy()
    test_model_frame.index = pd.MultiIndex.from_product([[test_date], test_model_frame.index], names=["date", "minute"])

    fair_x, fair_y = valid_fair_value_frame(train_model_frame, state_features)
    train_features = available_feature_columns(train_quote_frame, model_features, min_obs=MIN_FEATURE_OBS)
    train_touch = train_quote_frame.dropna(subset=["touch"])
    if len(fair_y) < MIN_TRAIN_OBS // 2 or len(train_touch) < MIN_TRAIN_OBS or train_touch["touch"].nunique() < 2 or not train_features:
        walk_forward_rows.append({"test_date": test_date, "status": "skipped_insufficient_train", "train_obs": len(train_touch), "test_obs": len(test_frame)})
        continue

    fair_model = make_fair_value_model()
    fair_model.fit(fair_x, fair_y)
    fair_edge_train = pd.Series(fair_model.predict(train_model_frame.reindex(columns=fair_x.columns).replace([np.inf, -np.inf], np.nan)), index=train_model_frame.index, name="fair_value_edge_bps")
    fair_edge_test = pd.Series(fair_model.predict(test_model_frame.reindex(columns=fair_x.columns).replace([np.inf, -np.inf], np.nan)), index=test_model_frame.index, name="fair_value_edge_bps")

    train_quote_frame["fair_value_edge_bps"] = fair_edge_train.reindex(train_quote_frame.index).to_numpy()
    test_frame["fair_value_edge_bps"] = fair_edge_test.reindex(test_frame.index).to_numpy()
    train_quote_frame["fair_value_edge_if_filled_bps"] = side_fair_value_edge_bps(train_quote_frame)
    test_frame["fair_value_edge_if_filled_bps"] = side_fair_value_edge_bps(test_frame)
    train_quote_frame["toxicity_residual_bps"] = train_quote_frame["markout_bps"] - train_quote_frame["fair_value_edge_if_filled_bps"]

    touch_model = make_touch_model()
    touch_model.fit(train_touch[train_features], train_touch["touch"])
    touch_prob = pd.Series(touch_model.predict_proba(test_frame.reindex(columns=train_features))[:, 1], index=test_frame.index, name="touch_prob")

    touched_train = train_quote_frame[train_quote_frame["touch"] == 1.0].dropna(subset=["markout_bps", "toxicity_residual_bps"])
    if len(touched_train) >= MIN_TRAIN_OBS // 4:
        raw_markout_model = make_ridge_model()
        raw_markout_model.fit(touched_train[train_features], touched_train["markout_bps"])
        predicted_raw_markout = pd.Series(raw_markout_model.predict(test_frame.reindex(columns=train_features)), index=test_frame.index, name="predicted_raw_markout_bps")

        residual_model = make_ridge_model()
        residual_model.fit(touched_train[train_features], touched_train["toxicity_residual_bps"])
        predicted_residual = pd.Series(residual_model.predict(test_frame.reindex(columns=train_features)), index=test_frame.index, name="predicted_toxicity_residual_bps")
    else:
        predicted_raw_markout = pd.Series(touched_train["markout_bps"].mean(), index=test_frame.index, name="predicted_raw_markout_bps")
        predicted_residual = pd.Series(touched_train["toxicity_residual_bps"].mean(), index=test_frame.index, name="predicted_toxicity_residual_bps")

    predicted_total_markout = test_frame["fair_value_edge_if_filled_bps"] + predicted_residual
    out = test_frame[["side", "side_sign", "quote_distance_bps", "touch", "markout_bps", "realized_quote_value_bps", "vol_scale", "fair_value_edge_bps", "fair_value_edge_if_filled_bps"]].copy()
    out["touch_prob"] = touch_prob
    out["predicted_raw_markout_bps"] = predicted_raw_markout
    out["predicted_standalone_value_bps"] = touch_prob * (predicted_raw_markout - MAKER_FEE_BPS)
    out["predicted_toxicity_residual_bps"] = predicted_residual
    out["predicted_total_markout_bps"] = predicted_total_markout
    out["predicted_fv_value_bps"] = touch_prob * (predicted_total_markout - MAKER_FEE_BPS)
    walk_forward_frames.append(out)

    touched_test = test_frame["touch"] == 1.0
    walk_forward_rows.append(
        {
            "test_date": test_date,
            "status": "fit",
            "train_dates": ",".join(train_dates),
            "train_obs": len(train_touch),
            "test_obs": len(test_frame),
            "feature_count": len(train_features),
            **touch_metrics(test_frame["touch"], touch_prob),
            **regression_metrics(test_frame.loc[touched_test, "markout_bps"], predicted_raw_markout.loc[touched_test], "raw_markout"),
            **regression_metrics(test_frame.loc[touched_test, "markout_bps"], predicted_total_markout.loc[touched_test], "fv_total_markout"),
        }
    )

walk_forward_summary = pd.DataFrame(walk_forward_rows)
walk_forward_quotes = pd.concat(walk_forward_frames).sort_index() if walk_forward_frames else pd.DataFrame(index=quote_model_table.index)
display(walk_forward_summary)
walk_forward_quotes.tail(20)

## All-Sample Research Fit

This section is intentionally labeled research fit. It is useful for diagnostics and coefficient inspection, but the walk-forward section is the relevant sanity check for policy selection.

In [ ]:
fair_x, fair_y = valid_fair_value_frame(model_panel, state_features)
research_fair_model = make_fair_value_model()
research_fair_model.fit(fair_x, fair_y)
research_fair_edge = pd.Series(research_fair_model.predict(model_panel.reindex(columns=fair_x.columns).replace([np.inf, -np.inf], np.nan)), index=model_panel.index, name="fair_value_edge_bps")

research_quotes = quote_model_table.copy()
research_quotes["fair_value_edge_bps"] = research_fair_edge.reindex(research_quotes.index).to_numpy()
research_quotes["fair_value_edge_if_filled_bps"] = side_fair_value_edge_bps(research_quotes)
research_quotes["toxicity_residual_bps"] = research_quotes["markout_bps"] - research_quotes["fair_value_edge_if_filled_bps"]

research_features = available_feature_columns(research_quotes, model_features, min_obs=MIN_FEATURE_OBS)
research_touch = research_quotes.dropna(subset=["touch"])
research_touch_model = make_touch_model()
research_touch_model.fit(research_touch[research_features], research_touch["touch"])
research_touch_prob = pd.Series(research_touch_model.predict_proba(research_quotes.reindex(columns=research_features))[:, 1], index=research_quotes.index, name="touch_prob")

research_touched = research_quotes[research_quotes["touch"] == 1.0].dropna(subset=["markout_bps", "toxicity_residual_bps"])
research_raw_markout_model = make_ridge_model()
research_raw_markout_model.fit(research_touched[research_features], research_touched["markout_bps"])
research_raw_markout_pred = pd.Series(research_raw_markout_model.predict(research_quotes.reindex(columns=research_features)), index=research_quotes.index, name="predicted_raw_markout_bps")

research_residual_model = make_ridge_model()
research_residual_model.fit(research_touched[research_features], research_touched["toxicity_residual_bps"])
research_residual_pred = pd.Series(research_residual_model.predict(research_quotes.reindex(columns=research_features)), index=research_quotes.index, name="predicted_toxicity_residual_bps")
research_total_markout_pred = research_quotes["fair_value_edge_if_filled_bps"] + research_residual_pred

research_quotes["touch_prob"] = research_touch_prob
research_quotes["predicted_raw_markout_bps"] = research_raw_markout_pred
research_quotes["predicted_standalone_value_bps"] = research_touch_prob * (research_raw_markout_pred - MAKER_FEE_BPS)
research_quotes["predicted_toxicity_residual_bps"] = research_residual_pred
research_quotes["predicted_total_markout_bps"] = research_total_markout_pred
research_quotes["predicted_fv_value_bps"] = research_touch_prob * (research_total_markout_pred - MAKER_FEE_BPS)

research_touched_mask = research_quotes["touch"] == 1.0
research_metrics = pd.DataFrame(
    [
        {
            **touch_metrics(research_quotes["touch"], research_quotes["touch_prob"]),
            **regression_metrics(research_quotes.loc[research_touched_mask, "markout_bps"], research_quotes.loc[research_touched_mask, "predicted_raw_markout_bps"], "raw_markout"),
            **regression_metrics(research_quotes.loc[research_touched_mask, "markout_bps"], research_quotes.loc[research_touched_mask, "predicted_total_markout_bps"], "fv_total_markout"),
        }
    ]
)

touch_coef = pd.DataFrame({"feature": research_features, "touch_coef": research_touch_model.named_steps["logit"].coef_[0]})
touch_coef["abs_touch_coef"] = touch_coef["touch_coef"].abs()
touch_coef["family"] = touch_coef["feature"].map(feature_family)
raw_markout_coef = pd.DataFrame({"feature": research_features, "raw_markout_coef": research_raw_markout_model.named_steps["ridge"].coef_})
raw_markout_coef["abs_raw_markout_coef"] = raw_markout_coef["raw_markout_coef"].abs()
raw_markout_coef["family"] = raw_markout_coef["feature"].map(feature_family)
residual_coef = pd.DataFrame({"feature": research_features, "residual_coef": research_residual_model.named_steps["ridge"].coef_})
residual_coef["abs_residual_coef"] = residual_coef["residual_coef"].abs()
residual_coef["family"] = residual_coef["feature"].map(feature_family)

display(research_metrics)
display(touch_coef.sort_values("abs_touch_coef", ascending=False).head(25))
display(raw_markout_coef.sort_values("abs_raw_markout_coef", ascending=False).head(25))
display(residual_coef.sort_values("abs_residual_coef", ascending=False).head(25))

## Standalone Quote Policy Scan

This is the direct answer to: can we get closer to a standalone quoting strategy before fair value and inventory? A threshold policy quotes only candidates whose predicted raw standalone EV clears the threshold. The walk-forward table is the table to trust first; the research table is optimistic.

In [ ]:
def standalone_threshold_scan(frame: pd.DataFrame, source: str) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    total_candidates = len(frame)
    for threshold in STANDALONE_THRESHOLDS_BPS:
        selected = frame[frame["predicted_standalone_value_bps"] >= threshold].copy()
        touched = selected[selected["touch"] == 1.0]
        rows.append(
            {
                "source": source,
                "threshold_bps": threshold,
                "candidate_count": len(selected),
                "selection_rate": len(selected) / total_candidates if total_candidates else np.nan,
                "predicted_ev_bps": selected["predicted_standalone_value_bps"].mean(),
                "realized_ev_bps": selected["realized_quote_value_bps"].mean(),
                "touch_rate": selected["touch"].mean(),
                "filled_count": len(touched),
                "mean_markout_if_touched": touched["markout_bps"].mean(),
                "adverse_touch_share": (touched["markout_bps"] < 0).mean() if len(touched) else np.nan,
            }
        )
    return pd.DataFrame(rows)


policy_frames = [standalone_threshold_scan(research_quotes, "research_all_sample")]
if not walk_forward_quotes.empty:
    policy_frames.insert(0, standalone_threshold_scan(walk_forward_quotes, "walk_forward"))
standalone_policy = pd.concat(policy_frames, ignore_index=True)

display(standalone_policy)

standalone_by_bucket = (
    (walk_forward_quotes if not walk_forward_quotes.empty else research_quotes)
    .groupby(["side", "quote_distance_bps"], as_index=False)
    .agg(
        rows=("touch", "count"),
        touch_rate=("touch", "mean"),
        predicted_standalone_value_bps=("predicted_standalone_value_bps", "mean"),
        realized_quote_value_bps=("realized_quote_value_bps", "mean"),
        predicted_raw_markout_bps=("predicted_raw_markout_bps", "mean"),
    )
    .sort_values(["side", "predicted_standalone_value_bps"], ascending=[True, False])
)
display(standalone_by_bucket)

## Inventory-Aware Quote Construction

The inventory policy shifts the score by penalizing quotes that add to existing inventory and rewarding quotes that reduce it. Positive inventory means we are long BTC exposure; bids add more long exposure, asks reduce it.

In [ ]:
policy_base = walk_forward_quotes.copy() if not walk_forward_quotes.empty else research_quotes.copy()
policy_source = "walk_forward" if not walk_forward_quotes.empty else "research_all_sample"

scenario_frames: list[pd.DataFrame] = []
for inventory_units in INVENTORY_UNITS:
    scenario = policy_base.copy()
    scenario["inventory_units"] = inventory_units
    scenario["inventory_penalty_bps"] = scenario["side_sign"] * inventory_units * INVENTORY_RISK_BPS_PER_UNIT * scenario["vol_scale"]
    scenario["inventory_edge_if_filled_bps"] = scenario["fair_value_edge_if_filled_bps"] + scenario["predicted_toxicity_residual_bps"] - MAKER_FEE_BPS - scenario["inventory_penalty_bps"]
    scenario["model_risk_haircut_bps"] = MODEL_RISK_HAIRCUT_BPS
    scenario["risk_adjusted_edge_if_filled_bps"] = scenario["inventory_edge_if_filled_bps"] - scenario["model_risk_haircut_bps"]
    scenario["quote_score_bps"] = scenario["touch_prob"] * scenario["risk_adjusted_edge_if_filled_bps"]
    scenario["quoteable"] = scenario["quote_score_bps"] > MIN_QUOTE_SCORE_BPS
    scenario_frames.append(scenario)

inventory_scores = pd.concat(scenario_frames).sort_index()
inventory_reset = inventory_scores.reset_index().reset_index(names="row_id")
best_idx = inventory_reset.groupby(["date", "minute", "side", "inventory_units"])["quote_score_bps"].idxmax()
best_inventory_quotes = inventory_reset.loc[best_idx].sort_values(["inventory_units", "date", "minute", "side"]).reset_index(drop=True)

inventory_summary = (
    best_inventory_quotes.groupby(["inventory_units", "side"], as_index=False)
    .agg(
        rows=("quote_score_bps", "count"),
        quoteable_share=("quoteable", "mean"),
        mean_selected_distance_bps=("quote_distance_bps", "mean"),
        mean_touch_prob=("touch_prob", "mean"),
        mean_quote_score_bps=("quote_score_bps", "mean"),
        mean_inventory_edge_if_filled_bps=("inventory_edge_if_filled_bps", "mean"),
        mean_risk_adjusted_edge_if_filled_bps=("risk_adjusted_edge_if_filled_bps", "mean"),
        realized_ev_bps=("realized_quote_value_bps", "mean"),
    )
)

display(inventory_summary)
display(best_inventory_quotes.head(40))

## Diagnostics

These plots show whether positive EV is coming from real quote economics, very sparse far-away fills, or inventory skew.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.lineplot(data=quote_baseline, x="quote_distance_bps", y="realized_quote_value_bps", hue="side", marker="o", ax=axes[0, 0])
axes[0, 0].axhline(0, color="black", linewidth=1)
axes[0, 0].set_title("Raw Passive Quote EV By Distance")
axes[0, 0].set_ylabel("bps per posted quote")

sns.lineplot(data=quote_baseline, x="quote_distance_bps", y="required_rebate_per_fill_bps", hue="side", marker="o", ax=axes[0, 1])
axes[0, 1].set_title("Maker Rebate Needed To Offset Touched Markout")
axes[0, 1].set_ylabel("bps per fill")

sensitivity_pivot = sensitivity_summary.pivot(index="touch_horizon_minutes", columns="markout_minutes", values="best_realized_quote_value_bps")
sns.heatmap(sensitivity_pivot, annot=True, fmt=".2f", center=0, cmap="vlag", ax=axes[1, 0])
axes[1, 0].set_title("Best Raw Bucket EV Across Horizon Grid")

sns.lineplot(data=standalone_policy, x="threshold_bps", y="realized_ev_bps", hue="source", marker="o", ax=axes[1, 1])
axes[1, 1].axhline(0, color="black", linewidth=1)
axes[1, 1].set_title("Realized EV After Standalone Predicted-EV Threshold")
axes[1, 1].set_ylabel("bps per selected quote")

plt.tight_layout()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.lineplot(data=inventory_summary, x="inventory_units", y="mean_selected_distance_bps", hue="side", marker="o", ax=axes[0])
axes[0].set_title("Selected Distance By Inventory")
sns.lineplot(data=inventory_summary, x="inventory_units", y="mean_quote_score_bps", hue="side", marker="o", ax=axes[1])
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Inventory-Aware Quote Score")
plt.tight_layout()

## Portfolio Manager Interpretation

The practical levers are ranked as follows.

1. Wider quote distances can move raw bucket EV toward positive, but fill rates fall quickly. Far-away positive buckets are research candidates, not proof of scalable PnL.

2. Lower fees do not solve the current baseline by themselves because the default fee is already zero. Existing negative touched markouts require a maker rebate equal to the negative markout per fill, which is usually much larger than realistic exchange rebates.

3. Lower latency helps only if it changes the fill set. In this bar-level notebook, latency is represented indirectly by the touch horizon. A production answer needs event-level order book replay with queue position, cancel latency, post-only rejections, and fill priority.

4. Standalone positive EV should come from a combination of toxicity filtering, wider distances, and quote/no-quote thresholds. A model that only quotes when predicted standalone EV is positive is the cleanest test, but it needs more out-of-sample days.

5. The inventory quote model is the next portfolio layer. It should not rescue bad quotes by hiding adverse selection; it should shift size and distance so quotes reduce risk when inventory is off target.

6. The default inventory score includes a model-risk haircut. If removing the haircut is necessary to make quotes look attractive, the strategy is not ready for production.

## Optional Save

Set `SAVE_OUTPUTS = True` to write baseline, sensitivity, walk-forward, standalone-policy, and inventory-policy tables.

In [ ]:
if SAVE_OUTPUTS:
    out_dir = FEATURE_ROOT / "inventory_quote_model" / f"bar_{BAR_MINUTES}m"
    out_dir.mkdir(parents=True, exist_ok=True)
    date_inventory.to_parquet(out_dir / "date_inventory.parquet")
    build_summary.to_parquet(out_dir / "build_summary.parquet")
    quote_baseline.to_parquet(out_dir / "quote_baseline.parquet")
    sensitivity_summary.to_parquet(out_dir / "sensitivity_summary.parquet")
    if not positive_buckets.empty:
        positive_buckets.to_parquet(out_dir / "positive_buckets.parquet")
    walk_forward_summary.to_parquet(out_dir / "walk_forward_summary.parquet")
    if not walk_forward_quotes.empty:
        walk_forward_quotes.to_parquet(out_dir / "walk_forward_quotes.parquet")
    research_metrics.to_parquet(out_dir / "research_metrics.parquet")
    standalone_policy.to_parquet(out_dir / "standalone_policy.parquet")
    standalone_by_bucket.to_parquet(out_dir / "standalone_by_bucket.parquet")
    inventory_summary.to_parquet(out_dir / "inventory_summary.parquet")
    best_inventory_quotes.to_parquet(out_dir / "best_inventory_quotes.parquet")
    print(f"wrote inventory quote outputs to {out_dir}")
else:
    print("SAVE_OUTPUTS is false; nothing written")